# Titanic 29

Titanic 28에 depth=4, l2_leaf_reg=20을 결합하면 일반화가 개선될 것이다.


Titanic 28의 FeaturePreprocessor(BASE_FEATURES)와 baseline_parameters를 그대로 사용한다.
새 피처/전처리/인코딩을 추가하거나 삭제하지 않는다. 각 fold에서 전처리를 새로 fit하고 validation에는 transform만 한다.
test는 최종 확률 예측에만 사용한다. 기본 bootstrap도 기존 설정을 유지한다.
Best Iteration은 0-based index + 1인 트리 개수이다. full train은 fold 중앙값으로 재학습한다.

**평가 해석:** CV validation으로 early stopping을 선택하므로 OOF AUC에는 선택 낙관성이 있다.
기존 25% holdout도 full-data CV와 겹치고 holdout early stopping에 재사용된다.
따라서 holdout은 28과 동일 방식의 보조 진단값이며 독립 test 성능이 아니다.
앙상블 weight의 같은 OOF 평가 역시 탐색 편향이 있다.
CV Std는 표본 표준편차(ddof=1), Train/Validation Gap은 holdout train AUC - holdout AUC이다.
CV Gap도 따로 표시한다. Public Score는 입력받거나 선택에 사용하지 않는다.

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from common.interaction_experiments import FeaturePreprocessor
from common.feature_experiments import baseline_parameters

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
submission_template = pd.read_csv('csv/submission.csv')

target_col = 'survived'
id_col = 'passengerid'
BASE_FEATURES = ['GenderClass', 'GenderIsChild', 'ClassIsChild', 'AgeBand']
X = train.drop(columns=[target_col, id_col])
y = train[target_col].astype('int8')
X_test_raw = test.drop(columns=[id_col])

assert X.columns.equals(X_test_raw.columns)
assert train[id_col].is_unique and test[id_col].is_unique
assert set(train[id_col]).isdisjoint(test[id_col])
train_part, valid_part = train_test_split(
    train, test_size=0.25, stratify=train[target_col], random_state=SEED
)
X_tr_raw = train_part.drop(columns=[target_col, id_col])
y_tr = train_part[target_col].astype('int8')
X_valid_raw = valid_part.drop(columns=[target_col, id_col])
y_valid = valid_part[target_col].astype('int8')
assert len(train_part) == 687 and len(valid_part) == 229

BASE_PARAMS = baseline_parameters()
assert BASE_PARAMS['random_state'] == SEED
print('Titanic 15 명시 파라미터:', BASE_PARAMS)
print('고정 피처:', BASE_FEATURES)
import sys
import catboost, sklearn
print("Runtime:", sys.version)
print("Versions:", catboost.__version__, sklearn.__version__, pd.__version__)
PARAMS = {
 '28': {'learning_rate': 0.03},
 '29': {'learning_rate': 0.03, 'depth': 4, 'l2_leaf_reg': 20},
 '30': {'learning_rate': 0.03, 'random_strength': 0.2, 'subsample': 0.8, 'rsm': 1.0},
 'A': {'learning_rate': 0.03, 'depth': 4, 'l2_leaf_reg': 20,
       'random_strength': 0.2, 'subsample': 0.8, 'rsm': 1.0},
}
SPLITS=list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X,y))
# Declare practical stability criteria before seeing results.
MIN_GAIN=0.001
MAX_STD_INCREASE=0.003
MAX_GAP_INCREASE=0.01
MAX_HOLDOUT_DROP=0.005
MIN_IMPROVED_FOLDS=3
MAX_SINGLE_FOLD_SHARE=0.60

Titanic 15 명시 파라미터: {'verbose': 0, 'random_state': 42, 'cat_features': [], 'allow_writing_files': False}
고정 피처: ['GenderClass', 'GenderIsChild', 'ClassIsChild', 'AgeBand']
Runtime: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Versions: 1.2.10 1.9.0 3.0.5


## Random State Audit

In [2]:
RANDOM_AUDIT = pd.DataFrame([
    ('Python random', 42, 'random.seed(42)'),
    ('NumPy', 42, 'np.random.seed(42)'),
    ('train_test_split', 42, 'explicit random_state'),
    ('StratifiedKFold', 42, 'shuffle=True, explicit random_state'),
    ('CatBoost', 42, 'baseline random_state alias -> random_seed'),
    ('RandomizedSearch', 'N/A', 'not used; deterministic grid only'),
], columns=['Component', 'Random State', 'Code basis'])
display(RANDOM_AUDIT)
assert RANDOM_AUDIT.loc[RANDOM_AUDIT['Random State'].ne('N/A'), 'Random State'].eq(42).all()
print('모든 사용 stochastic component의 seed가 코드에서 42로 고정되었습니다.')

모든 사용 stochastic component의 seed가 코드에서 42로 고정되었습니다.


,Component,Random State,Code basis
0,Python random,42,random.seed(42)
1,NumPy,42,np.random.seed(42)
2,train_test_split,42,explicit random_state
3,StratifiedKFold,42,"shuffle=True, explicit random_state"
4,CatBoost,42,baseline random_state alias -> random_seed
5,RandomizedSearch,N/A,not used; deterministic grid only


In [3]:
def prepare_fold(X_train_raw, X_valid_raw):
    prep = FeaturePreprocessor(BASE_FEATURES)
    X_train_model = prep.fit_transform(X_train_raw)
    X_valid_model = prep.transform(X_valid_raw)
    assert X_train_model.columns.equals(X_valid_model.columns)
    assert np.isfinite(X_train_model.to_numpy()).all()
    assert np.isfinite(X_valid_model.to_numpy()).all()
    return prep, X_train_model, X_valid_model


def fit_model(X_train_model, y_train, overrides=None, **fit_kwargs):
    params = BASE_PARAMS | (overrides or {})
    model = CatBoostClassifier(**params)
    model.fit(X_train_model, y_train, **fit_kwargs)
    return model


def auc_scores(model, X_train_model, y_train, X_valid_model, y_valid):
    positive_index = list(model.classes_).index(1)
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train_model)[:, positive_index])
    valid_auc = roc_auc_score(y_valid, model.predict_proba(X_valid_model)[:, positive_index])
    return train_auc, valid_auc, train_auc - valid_auc


def positive(model, values):
    assert model.get_all_params()['random_seed']==SEED
    return model.predict_proba(values)[:,list(model.classes_).index(1)]

def evaluate(label):
    rows, train_predictions=[],[]
    oof=np.full(len(y),np.nan)
    visits=np.zeros(len(y),dtype=int)
    params=PARAMS[label]|{'iterations':3000,'loss_function':'Logloss','eval_metric':'AUC'}
    for fold,(tr,va) in enumerate(SPLITS,1):
        assert not set(tr).intersection(va)
        _,xt,xv=prepare_fold(X.iloc[tr],X.iloc[va])
        model=fit_model(xt,y.iloc[tr],params,eval_set=(xv,y.iloc[va]),
                        early_stopping_rounds=150,use_best_model=True,verbose=False)
        pt,pv=positive(model,xt),positive(model,xv)
        oof[va]=pv
        visits[va]+=1
        train_predictions.append(pt)
        ta,vauc=roc_auc_score(y.iloc[tr],pt),roc_auc_score(y.iloc[va],pv)
        best_iteration=int(model.get_best_iteration())+1
        assert best_iteration==model.tree_count_ and 1<=best_iteration<=3000
        rows.append(dict(fold=fold,train_auc=ta,validation_auc=vauc,gap=ta-vauc,best_iteration=best_iteration))
        print(f"Titanic {label} Fold {fold}: AUC={vauc:.9f}, Best Iteration={best_iteration}")
    assert (visits==1).all() and np.isfinite(oof).all()
    folds=pd.DataFrame(rows)
    iterations=int(folds.best_iteration.median())
    assert 1<=iterations<=3000
    _,ht,hv=prepare_fold(X_tr_raw,X_valid_raw)
    hm=fit_model(ht,y_tr,params,eval_set=(hv,y_valid),early_stopping_rounds=150,
                 use_best_model=True,verbose=False)
    hp_train,hp_valid=positive(hm,ht),positive(hm,hv)
    ht_auc,hv_auc=roc_auc_score(y_tr,hp_train),roc_auc_score(y_valid,hp_valid)
    summary={
      'Model':label,'CV Mean':float(folds.validation_auc.mean()),
      'CV Std':float(folds.validation_auc.std(ddof=1)),'OOF AUC':float(roc_auc_score(y,oof)),
      'Holdout AUC':float(hv_auc),'Train/Validation Gap':float(ht_auc-hv_auc),
      'CV Gap':float(folds.gap.mean()),'Best Iteration Mean':float(folds.best_iteration.mean()),
      'Best Iteration Median':float(folds.best_iteration.median()),'Final Iterations':iterations,
      'Parameter':str(BASE_PARAMS|params|{'iterations':iterations})}
    print("Best iteration distribution:",folds.best_iteration.tolist())
    print("Full-train iterations = CV median:",iterations,"(no override)")
    display(folds)
    display(pd.DataFrame([summary]))
    if label=='28':
        np.testing.assert_allclose(summary['CV Mean'],0.914324942791762,atol=1e-12,rtol=0)
        np.testing.assert_allclose(summary['CV Std'],0.012961501591864434,atol=1e-12,rtol=0)
        assert iterations==20
        print("Titanic 28 recorded result reproduced.")
    return dict(summary=summary,folds=folds,oof=oof,train_predictions=train_predictions,
                holdout_train=hp_train,holdout_valid=hp_valid,iterations=iterations)

def full_prediction(label,evaluation):
    prep=FeaturePreprocessor(BASE_FEATURES)
    full=prep.fit_transform(X)
    test_model=prep.transform(X_test_raw)
    assert full.columns.equals(test_model.columns)
    params=PARAMS[label]|{'iterations':evaluation['iterations'],'loss_function':'Logloss','eval_metric':'AUC'}
    model=fit_model(full,y,params)
    assert model.tree_count_==evaluation['iterations']
    return positive(model,test_model)

def save_submission(predictions,number):
    assert submission_template.columns.tolist()==[id_col,target_col]
    assert len(submission_template)==len(test)
    assert submission_template[id_col].is_unique and submission_template[id_col].notna().all()
    assert set(submission_template[id_col])==set(test[id_col])
    assert predictions.shape==(len(test),)
    by_id=pd.Series(predictions,index=test[id_col].to_numpy())
    result=submission_template.copy(deep=True)
    result[target_col]=result[id_col].map(by_id)
    assert result.shape==submission_template.shape and result.columns.equals(submission_template.columns)
    pd.testing.assert_series_equal(result[id_col],submission_template[id_col],check_names=True)
    p=result[target_col].to_numpy()
    assert pd.api.types.is_float_dtype(result[target_col])
    assert np.isfinite(p).all() and result[target_col].between(0,1).all()
    assert ((p>0)&(p<1)).any(),'Hard labels forbidden'
    output_path=Path(f'titanic_{number}_result.csv')
    result.to_csv(output_path,index=False)
    saved=pd.read_csv(output_path)
    assert saved.shape==(len(test),2) and saved.columns.equals(submission_template.columns)
    pd.testing.assert_series_equal(saved[id_col],submission_template[id_col])
    assert saved[target_col].notna().all() and saved[target_col].between(0,1).all()
    assert pd.api.types.is_float_dtype(saved[target_col])
    assert ((saved[target_col]>0)&(saved[target_col]<1)).any()
    np.testing.assert_allclose(saved[target_col],p,rtol=1e-12,atol=1e-15)
    print("Submission assertions passed:",output_path,saved.shape)
    return result

In [4]:
evaluations={}
for label in ['28','29']:
    evaluations[label]=evaluate(label)

Titanic 28 Fold 1: AUC=0.908771930, Best Iteration=150
Titanic 28 Fold 2: AUC=0.929570303, Best Iteration=70
Titanic 28 Fold 3: AUC=0.895245360, Best Iteration=9
Titanic 28 Fold 4: AUC=0.919590643, Best Iteration=2
Titanic 28 Fold 5: AUC=0.918446479, Best Iteration=20
Best iteration distribution: [150, 70, 9, 2, 20]
Full-train iterations = CV median: 20 (no override)
Titanic 28 recorded result reproduced.
Titanic 29 Fold 1: AUC=0.901879699, Best Iteration=332
Titanic 29 Fold 2: AUC=0.922133232, Best Iteration=4
Titanic 29 Fold 3: AUC=0.896262395, Best Iteration=4
Titanic 29 Fold 4: AUC=0.906114925, Best Iteration=92
Titanic 29 Fold 5: AUC=0.911772184, Best Iteration=8
Best iteration distribution: [332, 4, 4, 92, 8]
Full-train iterations = CV median: 8 (no override)


,fold,train_auc,validation_auc,gap,best_iteration
0,1,0.943578,0.908772,0.034806,150
1,2,0.929983,0.929570,0.000413,70
2,3,0.923238,0.895245,0.027992,9
3,4,0.909791,0.919591,-0.009800,2
4,5,0.921543,0.918446,0.003097,20


,Model,CV Mean,CV Std,OOF AUC,Holdout AUC,Train/Validation Gap,CV Gap,Best Iteration Mean,Best Iteration Median,Final Iterations,Parameter
0,28,0.914325,0.012962,0.878288,0.906367,0.01912,0.011302,50.2,20.0,20,"{'verbose': 0, 'random_state': 42, 'cat_featur..."


,fold,train_auc,validation_auc,gap,best_iteration
0,1,0.934580,0.901880,0.032700,332
1,2,0.897480,0.922133,-0.024653,4
2,3,0.911972,0.896262,0.015710,4
3,4,0.918808,0.906115,0.012693,92
4,5,0.908722,0.911772,-0.003050,8


,Model,CV Mean,CV Std,OOF AUC,Holdout AUC,Train/Validation Gap,CV Gap,Best Iteration Mean,Best Iteration Median,Final Iterations,Parameter
0,29,0.907632,0.009901,0.869942,0.898845,0.031055,0.00668,88.0,8.0,8,"{'verbose': 0, 'random_state': 42, 'cat_featur..."


In [5]:
comparison=pd.DataFrame([evaluations[k]['summary'] for k in ['28','29']])
display(comparison)
print("CV Mean change vs Titanic 28:",evaluations['29']['summary']['CV Mean']-evaluations['28']['summary']['CV Mean'])
predictions=full_prediction('29',evaluations['29'])
result=save_submission(predictions,29)
REPORT={
 'summaries':[evaluations[k]['summary'] for k in ['28','29']],
 'fold_auc':{k:evaluations[k]['folds'].validation_auc.tolist() for k in evaluations},
 'best_iterations':{k:evaluations[k]['folds'].best_iteration.tolist() for k in evaluations},
 'final_iterations':{'29':evaluations['29']['iterations']},'random_state':SEED}

CV Mean change vs Titanic 28: -0.006692455777123851
Submission assertions passed: titanic_29_result.csv (393, 2)


,Model,CV Mean,CV Std,OOF AUC,Holdout AUC,Train/Validation Gap,CV Gap,Best Iteration Mean,Best Iteration Median,Final Iterations,Parameter
0,28,0.914325,0.012962,0.878288,0.906367,0.019120,0.011302,50.2,20.0,20,"{'verbose': 0, 'random_state': 42, 'cat_featur..."
1,29,0.907632,0.009901,0.869942,0.898845,0.031055,0.006680,88.0,8.0,8,"{'verbose': 0, 'random_state': 42, 'cat_featur..."


## 재현성
SEED=42, 고정 split/순서, CPU CatBoost, 결정적인 coarse grid를 사용한다. 별도 Python 프로세스 2회에서 모든 code cell 실행 후 결과와 CSV 해시를 비교한다. Jupyter에서도 Kernel Restart 후 Run All 가능하다. CatBoost 로그 파일 생성은 비활성화했다.

**실행 검증 완료:** 별도 Python 프로세스 2회 전체 실행에서 Fold AUC/iteration/선택 결과와 CSV SHA-256이 일치함: a28f8e1ec3b6abb29e94dc1aa138aefc25a4d4d6bd62848f8ac32d40fbedadcf